# Project 06: Stimulus-to-Response GLM — Starter Notebook

This notebook is a STARTER only: it loads the data and sets up a few
scaffolding pieces so every group starts from the same place. It does
not contain a solution. Read `../exercise_handout.md` (or the PDF/DOCX
version) in full before you start.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../shared/src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# these are the ONLY workshop-provided modules -- everything else is yours
import analysis
import plotting
from models import build_design_matrix, drift_basis, MAIN_EFFECT_NAMES, DRIFT_BASIS_NAMES

DATA_DIR = os.path.abspath("../../shared/data")
df = pd.read_csv(os.path.join(DATA_DIR, "trials.csv"))
df.head()


## 1. Look at the experiment design

Confirm you understand `trial_role` and how many trials are FIT vs
HOLDOUT before doing anything else. Remember: `trials.csv` is in
original session order -- do not shuffle it.


In [ ]:
df.groupby("trial_role").size()


## 2. Task T2 — confound check and basis choice

Check whether `orientation_deg` and `contrast_pct` are correlated with
`trial_index`. Plot the raw response against trial index.


In [ ]:
D45 = (df.orientation_deg == 45).astype(float)
D90 = (df.orientation_deg == 90).astype(float)
D135 = (df.orientation_deg == 135).astype(float)
log_contrast = np.log2(df.contrast_pct / 12.5)
t_frac = df.trial_index / len(df)

for name, col in [("D45", D45), ("D90", D90), ("D135", D135), ("log_contrast", log_contrast)]:
    print(name, np.corrcoef(col, t_frac)[0, 1])


In [ ]:
fig = plotting.plot_response_vs_trial(df.trial_index, df.response, df.trial_role)
plt.show()


**TODO:** based on what you see above, decide on a basis for the drift
term $X_2$ (Section 3.1 of the handout). `models.drift_basis` provides
ONE reasonable basis -- read its docstring, and decide for yourselves
whether it (or something else) is the right choice given what your plot
actually shows.


## 3. Task T1 — derive and verify the omitted-variable-bias formula

Work through Section 3.2's derivation on paper first. Then use
`analysis.omitted_variable_bias` to check your Section 3.2 hand-
calculation, and write your OWN verification here: (a) the repeated-
noise-draw average-bias check, and (b) the exactly-zero-bias orthogonal
check. Do not skip either -- they are graded, and they are also the
fastest way to catch a sign error before it contaminates every later step.


In [ ]:
# Hand-check from Section 3.2: X1 = [1,2,3,4]' (no intercept), X2 = [1,1,2,3]', beta2 = 5
X1_toy = np.array([[1.0], [2.0], [3.0], [4.0]])
X2_toy = np.array([[1.0], [1.0], [2.0], [3.0]])
beta2_toy = np.array([5.0])
print(analysis.omitted_variable_bias(X1_toy, X2_toy, beta2_toy))
# TODO: confirm this matches your own by-hand arithmetic

# TODO: your own repeated-noise-draw verification
# TODO: your own exactly-zero-bias orthogonal-design verification


## 4. Task T3 — fit both models

`analysis.fit_ols` returns an `OLSFit` with `.beta`, `.se`,
`.residuals`, `.cov_beta`, and more -- read its docstring.
`models.build_design_matrix` builds either the naive or the full design
matrix depending on `include_drift_basis`.


In [ ]:
df_fit = df[df.trial_role == "fit"].reset_index(drop=True)
df_holdout = df[df.trial_role == "holdout"].reset_index(drop=True)

# TODO: build X_naive and X_full for df_fit, fit both with analysis.fit_ols,
# and compare to the Section 3.2 bias prediction (using your OWN fitted
# drift coefficients from the full model as your stand-in for beta_2 true)


## 5. Task T4 — residual diagnostics and nested F-test

`analysis.residual_autocorrelation` and `analysis.nested_f_test` are
provided as building blocks -- read their docstrings before assuming
they do exactly what you want for your own chosen lags/models.


In [ ]:
# TODO: your Task T4 pipeline here


## 6. Task T5 — contrasts

`analysis.evaluate_contrast` takes a contrast VECTOR (same length as the
model's number of coefficients) and an `OLSFit`. `analysis.
residual_bootstrap_ci` gives a resampling-based uncertainty range for
any contrast.


In [ ]:
# TODO: your Task T5 pipeline here (at least 3 contrasts, both models)


## 7. Task T6 — holdout validation

Remember: coefficients must come ONLY from FIT trials. Use
`analysis.predict` and `analysis.holdout_metrics`.


In [ ]:
# TODO: your Task T6 pipeline here


## 8. Next steps

Once your pipeline runs end to end on this notebook, turn it into
proper, tested functions in your own modules (or extend `analysis.py`
directly) rather than leaving your final analysis only in notebook
cells -- the submission checklist (handout Section 9) requires code that
runs end-to-end from the package root.
